# FlexCNN Medical Physics - Notebook Template

This notebook demonstrates how to use the modular parameter system for FlexCNN.

**Key Features:**
- Import default parameters from `user_params.py`
- Override specific parameters without modifying source files
- Rebuild configurations after parameter changes
- Reload package code without kernel restart using `reload_package()`

## Setup Environment

In [ ]:
import os
import sys
import shutil

def _detect_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

IN_COLAB = _detect_colab()

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    
    # Local directory for module access
    local_project_dir = "/content/Colab/Working"
    os.makedirs(local_project_dir, exist_ok=True)
    
    # Drive mount location
    drive_project_dir = "/content/drive/MyDrive/Colab/Working"
    
    # Copy modules from Drive to local
    for filename in ("build_dicts.py", "script_setup.py", "user_params.py"):
        src = os.path.join(drive_project_dir, filename)
        dst = os.path.join(local_project_dir, filename)
        if os.path.isfile(src):
            shutil.copy2(src, dst)
    
    # Add local directory to sys.path
    if local_project_dir not in sys.path:
        sys.path.insert(0, local_project_dir)
else:
    project_dir = os.getcwd()
    if project_dir not in sys.path:
        sys.path.insert(0, project_dir)

from script_setup import (
    sense_colab, sense_device, install_packages,
    setup_colab_environment, setup_local_environment,
    reload_package, refresh_repo
)

IN_COLAB = sense_colab()
print(f"Running in {'Colab' if IN_COLAB else 'Local'} environment")

Running in Local environment


In [5]:
# Setup environment (run once per session)
install_packages(IN_COLAB, ray_version=None)

if IN_COLAB:
    setup_colab_environment(
        github_username='petercl8',
        repo_name='FlexCNN_for_Medical_Physics',
        skip_git_update=False,
        force_fresh_clone=False  # Set to True if you get import errors after updating GitHub
    )
else:
    # Local setup - use 'walk' mode for fast reload
    setup_local_environment(
        repo_name='FlexCNN_for_Medical_Physics',
        mode='walk',
    )
# Test resources (function now available in global namespace)
list_compute_resources()

🖥️  Local environment detected. Installing CUDA-enabled PyTorch...
📦 Installing PyTorch with CUDA (cu124)...
✅ PyTorch installation complete.
📦 Installing other packages...
✅ Other packages installation complete.

CUDA Diagnostic Information:
✅ PyTorch version: 2.6.0+cu124
✅ CUDA available: True
✅ CUDA device count: 1
   GPU 0: NVIDIA GeForce RTX 3080 Ti Laptop GPU

📂 Added C:\Users\Peter Lindstrom\Desktop\FlexCNN_for_Medical_Physics to sys.path
📦 Loading FlexCNN_for_Medical_Physics package (walk mode)...


C:\Users\Peter Lindstrom\AppData\Roaming\jupyterlab-desktop\jlab_server\Lib\site-packages\hyperopt\atpe.py:19: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


✨ Injecting all symbols into global namespace...
✅ Setup complete: 246 symbols loaded into globals.
CPUs available: 20
GPUs available: 1
  GPU 0: NVIDIA GeForce RTX 3080 Ti Laptop GPU


## Load Default Parameters

In [ ]:
# Import after environment setup (build_dicts depends on package being importable)
from build_dicts import build_all_dicts
from user_params import get_params

# Get default parameters
params = get_params()

# Display some key parameters
print(f"Run mode: {params['run_mode']}")
print(f"Network type: {params['network_type']}")
print(f"Train SI: {params['train_SI']}")
print(f"Gen sino size: {params['gen_sino_size']}")
print(f"Gen image size: {params['gen_image_size']}")

[INFO] Plot mode: inline - using interactive backend: module://matplotlib_inline.backend_inline
Run mode: train
Network type: ACT
Train SI: True
Gen sino size: 180
Gen image size: 180


## Override Parameters (Optional)

Modify parameters as needed without changing source files:

In [ ]:
# Example: Change run mode and network type
params['run_mode'] = 'train'
params['network_type'] = 'ACT'
params['train_checkpoint_file'] = 'my-custom-checkpoint'

print(f"Updated run mode: {params['run_mode']}")
print(f"Updated network type: {params['network_type']}")
print(f"Updated epochs: {params['train_epochs']}")

Updated run mode: train
Updated network type: ACT
Updated epochs: 100


## Setup Paths and Device

In [ ]:
# Set main project directory (function available from environment setup)
project_dirPath = params['project_dirPath'] = setup_project_dirs(
    IN_COLAB,
    params['project_local_dirPath'],
    params['project_colab_dirPath'],
    mount_colab_drive=True
)

# Set device
device = params['device_opt'] = sense_device(device=params['device_opt'])

print(f"Project directory: {project_dirPath}")
print(f"Device: {device}")

Project directory: C:\Users\Peter Lindstrom\My Drive (lindstrom.peter@gmail.com)\Colab\Working
Device: cuda


## Build Configuration Dictionaries

In [ ]:
# Build all dictionaries
all_dicts = build_all_dicts(params)

# Extract main dictionaries
config = all_dicts['config']
paths = all_dicts['paths']
settings = all_dicts['settings']
base_dirs = all_dicts['base_dirs']
tune_opts = all_dicts['tune_opts']
test_opts = all_dicts['test_opts']

print("✅ Configuration dictionaries built successfully!")
print(f"Config keys: {list(config.keys())}")

✅ Configuration dictionaries built successfully!
Config keys: ['SI_alpha_min', 'SI_dropout', 'SI_exp_kernel', 'SI_fixedScale', 'SI_gen_fill', 'SI_gen_final_activ', 'SI_gen_hidden_dim', 'SI_gen_mult', 'SI_gen_neck', 'SI_gen_z_dim', 'SI_half_life_examples', 'SI_layer_norm', 'SI_learnedScale_init', 'SI_normalize', 'SI_output_scale_lr_mult', 'SI_pad_mode', 'SI_skip_mode', 'SI_stats_criterion', 'batch_base2_exponent', 'gen_b1', 'gen_b2', 'gen_image_channels', 'gen_image_size', 'gen_lr', 'gen_sino_channels', 'gen_sino_size', 'network_type', 'sup_base_criterion', 'train_SI']


# Visualizations

## 288x288 Networks

In [ ]:
act_sino_array_name = 'train-highCountSino-320x257.npy'
act_image_array_name ='train-actMap.npy'
recon1_array = 'train-highCountImage.npy'
indexes = [100]
checkpoint_name = 'checkpoint-ACT-288-padZeros-tunedSSIM-100epochs'
network_type = 'ACT'
fig_size = 1

config_ACT_SI = { # 288x288, tuned SSIM, pad_type='zeros', interemediate size = None, horiz_pool=1, vert_pool=1
  "SI_alpha_min": -1,
  "SI_dropout": False,
  "SI_exp_kernel": 3,
  "SI_fixedScale": 1,
  "SI_gen_fill": 0,
  "SI_gen_final_activ": "LeakyReLU",
  "SI_gen_hidden_dim": 10,
  "SI_gen_mult": 1.5012782419950113,
  "SI_gen_neck": "narrow",
  "SI_gen_z_dim": 872,
  "SI_half_life_examples": -1,
  "SI_layer_norm": "instance",
  "SI_learnedScale_init": 7.305980552864529,
  "SI_normalize": False,
  "SI_output_scale_lr_mult": 1.6943444827125673,
  "SI_pad_mode": "zeros",
  "SI_skip_mode": "conv",
  "SI_stats_criterion": -1,
  "batch_base2_exponent": 7,
  "gen_b1": 0.3600790033157822,
  "gen_b2": 0.6033159868492163,
  "gen_image_channels": 1,
  "gen_image_size": 180,
  "gen_lr": 0.0024018267054557695,
  "gen_sino_channels": 3,
  "gen_sino_size": 288, # 288
  "network_type": "ACT",
  "sup_base_criterion": "MSELoss",
  "train_SI": True
}

device = "cpu"

PlotPhantomRecons(indexes, checkpoint_name, network_type, 
                  config_ACT_SI, paths, fig_size, device, settings,
                  outputs_to_plot = None,
                  act_image_array_name = act_image_array_name,
                  act_sino_array_name = act_sino_array_name, 
                  atten_image_array_name = None,
                  atten_sino_array_name = None, 
                  recon1_array_name = recon1_array, 
                  recon2_array_name = None)

'''
PlotPhantomRecons(indexes, checkpoint_name, network_type, 
                      config, paths, fig_size, device, settings, outputs_to_plot=None,
                      act_image_array_name, act_sino_array_name, atten_image_array_name=None,
                      atten_sino_array_name=None, recon1_array_name=recon1_array, recon2_array_name=None)
'''


C:\Users\Peter Lindstrom\Desktop\FlexCNN_for_Medical_Physics\FlexCNN_for_Medical_Physics\classes\dataset_classes.py:105: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:209.)
  act_image_multChannel = torch.from_numpy(np.ascontiguousarray(act_image_array[index,:])).float() if act_image_array is not None else None


Shape: torch.Size([1, 1, 180, 180]) // Min: 0.0 // Max: 59.88399887084961 // Mean: 2.0385310649871826 // Mean Sum (per image): 66048.40625 // Sum (a single image): 66048.40625
Shape: torch.Size([1, 1, 180, 180]) // Min: -0.3122905194759369 // Max: 261.8094482421875 // Mean: 10.081600189208984 // Mean Sum (per image): 326643.84375 // Sum (a single image): 326643.84375


# Reload Package After Code Changes

If you modify package code, use this cell to reload without restarting the kernel:

In [3]:
# Reload package after making code changes
reload_package()  # Reloads from local repo

# Or use refresh_repo() to pull from git and reinstall
#refresh_repo(IN_COLAB=IN_COLAB, force_fresh_clone=False)

# Rebuild dictionaries with updated code
all_dicts = build_all_dicts(params)
config = all_dicts['config']
paths = all_dicts['paths']
settings = all_dicts['settings']

print("✅ Package reloaded and dictionaries rebuilt!")

NameError: name 'reload_package' is not defined

# Example: Quick Parameter Changes

Example of quickly changing parameters and re-running:

In [ ]:
# Quick experiment: visualize data
params['run_mode'] = 'visualize'
params['visualize_batch_size'] = 10
params['visualize_shuffle'] = True

# Rebuild and run
all_dicts = build_all_dicts(params)
run_pipeline(
    config=all_dicts['config'],
    paths=all_dicts['paths'],
    settings=all_dicts['settings'],
    tune_opts=all_dicts['tune_opts'],
    base_dirs=all_dicts['base_dirs'],
    test_opts=all_dicts['test_opts'],
)

# Run Pipeline

In [ ]:
# Run the pipeline (function available from environment setup)
run_pipeline(
    config=config,
    paths=paths,
    settings=settings,
    tune_opts=tune_opts,
    base_dirs=base_dirs,
    test_opts=test_opts,
)